# GemmaFit v4.1 Senior Evidence Router - FunctionGemma 270M

Colab-resumable training notebook for the Senior Hero v4.1 router.

Goal: train a small local FunctionGemma router that maps `objective movement evidence + subjective self-report evidence + capability_contract + evidence_ledger` to exactly one bounded tool call.

Important boundaries:
- No raw video, raw landmarks, force, GRF, joint moment, EMG, heart-rate, fall-risk, sarcopenia, rehab-progress, diagnosis, or clinical labels.
- C++/Kotlin deterministic gates remain authoritative.
- LiteRT conversion is intentionally a separate smoke-gated step, not mixed into this training runtime.


In [ ]:
# Phase -1 / 0: Drive mount, paths, resume helpers
from pathlib import Path
import glob
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time

try:
    from google.colab import drive, userdata  # type: ignore
    if not Path('/content/drive').exists() or not any(Path('/content/drive').iterdir()):
        drive.mount('/content/drive')
except Exception as exc:
    userdata = None
    print('Not running in Colab or Drive already unavailable:', exc)

RUN_SUFFIX = 'gemmafit_v4_1_senior_router'
ARTIFACT_PREFIX = 'gemmafit-v4-1-senior-router'
MODEL_TAG = os.environ.get('MODEL_TAG', 'functiongemma-270m-it')
BASE_MODEL = os.environ.get('BASE_MODEL', 'google/functiongemma-270m-it')
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/BKTTT429/GemmaFit.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', 'main')
RUN_NAME = f'{MODEL_TAG}_{RUN_SUFFIX}'

DRIVE_ROOT = Path('/content/drive/MyDrive/GemmaFit_train')
REPO_DIR = Path('/content/GemmaFit')
OUT_ROOT = DRIVE_ROOT / 'trained_outputs'
PHASE_DIR = OUT_ROOT / f'PHASE_DONE_{RUN_NAME}'
RUN_STATE = OUT_ROOT / f'RUN_STATE_{RUN_NAME}.json'
RUN_EVENTS = OUT_ROOT / f'RUN_EVENTS_{RUN_NAME}.jsonl'
DISCONNECT_POINTS = OUT_ROOT / f'DISCONNECT_POINTS_{RUN_NAME}.jsonl'
DONE_MARKER = OUT_ROOT / f'TRAINING_DONE_{RUN_NAME}.json'
DATASET = REPO_DIR / 'finetune/data/gemmafit_v4_senior_evidence_router.json'
EVAL_REPORT = REPO_DIR / 'finetune/metrics/tool_call_eval_v4_senior.json'
ADAPTER_DIR = OUT_ROOT / 'adapter' / f'{RUN_NAME}_adapter'
CKPT_DIR = OUT_ROOT / 'checkpoints' / RUN_NAME
MERGED_DIR = OUT_ROOT / 'merged_fp16' / f'{RUN_NAME}_merged_fp16'
LOG_DIR = OUT_ROOT / 'logs' / RUN_NAME
TARGET_LITERTLM = DRIVE_ROOT / f'{ARTIFACT_PREFIX}.litertlm'
REPO_DONE_COPY = REPO_DIR / 'finetune/metrics/training_done_v4_1.json'
REPO_RUN_STATE_COPY = REPO_DIR / 'finetune/metrics/run_state_v4_1.json'

for path in [OUT_ROOT, PHASE_DIR, CKPT_DIR, ADAPTER_DIR.parent, MERGED_DIR.parent, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def utc_now():
    return time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass
    os.replace(tmp, path)

def append_jsonl(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(payload, ensure_ascii=False) + '\n')
        f.flush()
        try:
            os.fsync(f.fileno())
        except OSError:
            pass

def record_event(phase, event, **data):
    payload = {'ts': utc_now(), 'run_name': RUN_NAME, 'phase': phase, 'event': event, **data}
    append_jsonl(RUN_EVENTS, payload)
    return payload

def latest_checkpoint(path=CKPT_DIR):
    path = Path(path)
    ckpts = []
    for item in path.glob('checkpoint-*'):
        if item.is_dir():
            try:
                step = int(item.name.split('-')[-1])
            except ValueError:
                step = -1
            ckpts.append((step, item.stat().st_mtime, str(item)))
    if not ckpts:
        return None
    return sorted(ckpts, key=lambda x: (x[0], x[1]), reverse=True)[0][2]

def phase_done(phase):
    return (PHASE_DIR / f'{phase}.json').exists()

def mark_phase_done(phase, outputs=None, next_phase=None):
    payload = {
        'phase': phase,
        'status': 'done',
        'timestamp': utc_now(),
        'run_name': RUN_NAME,
        'outputs': outputs or [],
        'next_phase': next_phase,
    }
    atomic_write_json(PHASE_DIR / f'{phase}.json', payload)
    write_run_state(phase, 'done', next_phase=next_phase, outputs=outputs or [], last_completed_phase=phase)
    return payload

def write_run_state(phase, status, next_phase=None, **data):
    payload = {
        'version': 'v4_1_senior_evidence_router',
        'run_name': RUN_NAME,
        'run_suffix': RUN_SUFFIX,
        'updated_at': utc_now(),
        'phase': phase,
        'status': status,
        'next_phase': next_phase,
        'latest_checkpoint': latest_checkpoint(CKPT_DIR),
        'dataset_path': str(DATASET),
        'adapter_path': str(ADAPTER_DIR),
        'merged_hf_path': str(MERGED_DIR),
        **data,
    }
    atomic_write_json(RUN_STATE, payload)
    record_event(phase, f'state_{status}', next_phase=next_phase)
    return payload

def record_disconnect_point(phase, note, **data):
    payload = {'ts': utc_now(), 'run_name': RUN_NAME, 'phase': phase, 'note': note, **data}
    append_jsonl(DISCONNECT_POINTS, payload)
    record_event(phase, 'safe_resume_point', note=note, **data)
    return payload

def last_completed_phase():
    files = sorted(PHASE_DIR.glob('*.json'), key=lambda x: x.stat().st_mtime, reverse=True)
    if not files:
        return None
    return files[0].stem

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def has_nonzero_safetensors(path):
    path = Path(path)
    return path.exists() and any(p.suffix == '.safetensors' and p.stat().st_size > 0 for p in path.glob('*.safetensors'))

print('=== Resume diagnostic ===')
print('Run name              :', RUN_NAME)
print('Base model            :', BASE_MODEL)
print('Done marker           :', DONE_MARKER, DONE_MARKER.exists())
print('Last completed phase  :', last_completed_phase())
print('Latest checkpoint     :', latest_checkpoint(CKPT_DIR))
print('Adapter exists        :', ADAPTER_DIR.exists())
print('Merged HF exists      :', MERGED_DIR.exists(), 'nonzero safetensors=', has_nonzero_safetensors(MERGED_DIR))
print('Expected LiteRT output:', TARGET_LITERTLM)
record_event('0_resume_diagnostic', 'printed', latest_checkpoint=latest_checkpoint(CKPT_DIR), last_completed_phase=last_completed_phase())


In [ ]:
# Phase 1: install training dependencies
# Keep LiteRT conversion deps out of this runtime. Conversion gets its own notebook/runtime.
INSTALL_DEPS = os.environ.get('INSTALL_DEPS', '1') == '1'
if phase_done('1_install_deps'):
    print('Phase 1 already done')
elif INSTALL_DEPS:
    cmd = [
        sys.executable, '-m', 'pip', 'install', '-U',
        'transformers>=4.51.3',
        'datasets>=2.20.0',
        'accelerate>=0.34.0',
        'peft>=0.13.0',
        'safetensors>=0.4.5',
        'huggingface_hub>=0.24.0',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)
    mark_phase_done('1_install_deps', next_phase='2_repo_dataset')
    record_disconnect_point('1_install_deps', 'deps_installed')
    print('If packages replaced loaded modules, restart runtime, rerun Phase -1/0, then continue at Phase 2.')
else:
    print('INSTALL_DEPS=0; skipping pip install')
    mark_phase_done('1_install_deps', next_phase='2_repo_dataset')


In [ ]:
# Phase 2 / 3: repo checkout, v4.1 dataset generation, strict schema gates
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
else:
    os.chdir(REPO_DIR)
    subprocess.call(['git', 'fetch', '--all', '--prune'])
    subprocess.call(['git', 'pull', '--ff-only'])

os.chdir(REPO_DIR)
print('Repo:', REPO_DIR)
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

subprocess.check_call([sys.executable, 'finetune/data/generate_v4_senior_router.py', '--validate'])
subprocess.check_call([sys.executable, 'finetune/eval_v4_senior_router.py', '--dataset', str(DATASET), '--strict'])

dataset_sha256 = file_sha256(DATASET)
payload = json.loads(DATASET.read_text(encoding='utf-8'))
train_rows = len(payload.get('train', []))
validation_rows = {k: len(v) for k, v in payload.get('validation', {}).items()}
record_event('3_dataset_validation', 'dataset_validated', dataset=str(DATASET), sha256=dataset_sha256, train_rows=train_rows, validation_rows=validation_rows)
record_disconnect_point('3_dataset_validation', 'dataset_validated', dataset_path=str(DATASET), dataset_sha256=dataset_sha256)
mark_phase_done('3_dataset_validation', outputs=[str(DATASET), str(EVAL_REPORT)], next_phase='4_dataset_preview')
print(json.dumps({'sha256': dataset_sha256, 'train_rows': train_rows, 'validation_rows': validation_rows}, indent=2))


In [ ]:
# Phase 4: dataset preview and row-family audit
from collections import Counter
payload = json.loads(DATASET.read_text(encoding='utf-8'))
rows = payload['train'] + [row for split in payload['validation'].values() for row in split]
counts = Counter(row['row_type'] for row in rows)
print('Dataset version:', payload.get('version'))
print('Row counts:')
for key, value in counts.most_common():
    print(f'  {key:28s} {value}')

for wanted in ['subjective_checkin_prompt', 'subjective_checkin_record', 'persona_report', 'unsupported_question']:
    sample = next(row for row in rows if row['row_type'] == wanted)
    print('\n=== SAMPLE', wanted, '===')
    print('USER:')
    print(sample['messages'][1]['content'][:1400])
    print('ASSISTANT:')
    print(sample['messages'][2]['content'][:1400])

mark_phase_done('4_dataset_preview', outputs=[], next_phase='5_model_preflight')


In [ ]:
# Phase 5: model/tokenizer preflight and token-length check
try:
    token = os.environ.get('HF_TOKEN') or (userdata.get('HF_TOKEN') if userdata else None) or (userdata.get('HUGGINGFACE_TOKEN') if userdata else None)
    if token:
        os.environ['HF_TOKEN'] = token
        from huggingface_hub import login
        login(token=token, add_to_git_credential=False)
        print('HF token loaded from env/userdata')
except Exception as exc:
    print('HF login skipped:', exc)

import torch
from datasets import Dataset
from transformers import AutoTokenizer

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

MAX_SEQ_LENGTH = int(os.environ.get('MAX_SEQ_LENGTH', '1536'))
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def render_messages(messages, include_assistant=True):
    selected = messages if include_assistant else messages[:-1]
    if getattr(tokenizer, 'chat_template', None):
        try:
            return tokenizer.apply_chat_template(selected, tokenize=False, add_generation_prompt=not include_assistant)
        except Exception:
            pass
    chunks = []
    for msg in selected:
        chunks.append(f"<{msg['role']}>\n{msg['content']}\n")
    if not include_assistant:
        chunks.append('<assistant>\n')
    return ''.join(chunks)

payload = json.loads(DATASET.read_text(encoding='utf-8'))
lengths = []
for row in payload['train'][:500] + [r for split in payload['validation'].values() for r in split[:20]]:
    text = render_messages(row['messages'], include_assistant=True)
    lengths.append(len(tokenizer(text, add_special_tokens=False)['input_ids']))
print('Token length sample:', {'max': max(lengths), 'avg': round(sum(lengths) / len(lengths), 1), 'limit': MAX_SEQ_LENGTH})
if max(lengths) > MAX_SEQ_LENGTH:
    print('WARNING: some rows exceed MAX_SEQ_LENGTH and will be truncated.')
mark_phase_done('5_model_preflight', outputs=[], next_phase='6_training')
record_disconnect_point('5_model_preflight', 'tokenizer_ready', max_seq_length=MAX_SEQ_LENGTH, max_sample_tokens=max(lengths))


In [ ]:
# Phase 6: LoRA SFT training with checkpoint resume
# This cell trains by default. Set RUN_TRAINING=False for a dry notebook pass.
RUN_TRAINING = os.environ.get('RUN_TRAINING', '1') == '1'
MAX_STEPS = int(os.environ.get('MAX_STEPS', '600'))
SAVE_STEPS = int(os.environ.get('SAVE_STEPS', '50'))
EVAL_STEPS = int(os.environ.get('EVAL_STEPS', '50'))
LOGGING_STEPS = int(os.environ.get('LOGGING_STEPS', '10'))
TRAIN_BATCH_SIZE = int(os.environ.get('TRAIN_BATCH_SIZE', '4'))
GRAD_ACCUM = int(os.environ.get('GRAD_ACCUM', '4'))
LEARNING_RATE = float(os.environ.get('LEARNING_RATE', '2e-4'))
LORA_R = int(os.environ.get('LORA_R', '16'))
LORA_ALPHA = int(os.environ.get('LORA_ALPHA', '32'))
LORA_DROPOUT = float(os.environ.get('LORA_DROPOUT', '0.05'))

if not RUN_TRAINING:
    print('RUN_TRAINING=False; skipping training')
    record_event('6_training', 'skipped_by_flag')
elif phase_done('7_save_adapter') and ADAPTER_DIR.exists():
    print('Adapter already saved:', ADAPTER_DIR)
else:
    import torch
    from datasets import Dataset
    from transformers import AutoModelForCausalLM, DataCollatorForSeq2Seq, Trainer, TrainingArguments
    from peft import LoraConfig, TaskType, get_peft_model

    write_run_state('6_training', 'started', next_phase='7_save_adapter', checkpoint=latest_checkpoint(CKPT_DIR))
    payload = json.loads(DATASET.read_text(encoding='utf-8'))

    def to_pairs(items):
        return [{'messages': row['messages'], 'row_type': row.get('row_type', '')} for row in items]

    train_raw = Dataset.from_list(to_pairs(payload['train']))
    eval_source = []
    for split_name in ['care_log', 'subjective', 'persona', 'refusal', 'memory', 'adversarial']:
        eval_source.extend(payload['validation'].get(split_name, [])[:60])
    eval_raw = Dataset.from_list(to_pairs(eval_source))

    def tokenize_example(example):
        full_text = render_messages(example['messages'], include_assistant=True)
        prompt_text = render_messages(example['messages'], include_assistant=False)
        full = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LENGTH, add_special_tokens=False)
        prompt = tokenizer(prompt_text, truncation=True, max_length=MAX_SEQ_LENGTH, add_special_tokens=False)
        labels = list(full['input_ids'])
        prompt_len = min(len(prompt['input_ids']), len(labels))
        labels[:prompt_len] = [-100] * prompt_len
        full['labels'] = labels
        return full

    train_ds = train_raw.map(tokenize_example, remove_columns=train_raw.column_names, desc='tokenize train')
    eval_ds = eval_raw.map(tokenize_example, remove_columns=eval_raw.column_names, desc='tokenize eval')

    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
    )
    model.config.use_cache = False

    lora_targets = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    try:
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=lora_targets,
        )
        model = get_peft_model(model, lora_config)
    except Exception as exc:
        print('Default target_modules failed, retrying all-linear:', exc)
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules='all-linear',
        )
        model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    args = TrainingArguments(
        output_dir=str(CKPT_DIR),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        max_steps=MAX_STEPS,
        warmup_ratio=0.03,
        logging_steps=LOGGING_STEPS,
        save_strategy='steps',
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        eval_strategy='steps',
        eval_steps=EVAL_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        bf16=(dtype == torch.bfloat16),
        fp16=(dtype == torch.float16),
        report_to='none',
        remove_unused_columns=False,
        gradient_checkpointing=True,
    )
    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds, data_collator=collator)

    resume = latest_checkpoint(CKPT_DIR)
    record_event('6_training', 'trainer_start', resume_from_checkpoint=resume, max_steps=MAX_STEPS)
    result = trainer.train(resume_from_checkpoint=resume)
    trainer.save_state()
    metrics = result.metrics or {}
    atomic_write_json(LOG_DIR / 'train_metrics.json', metrics)
    record_event('6_training', 'trainer_done', metrics=metrics)
    record_disconnect_point('6_training', 'trainer_completed', checkpoint=latest_checkpoint(CKPT_DIR))

    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    mark_phase_done('7_save_adapter', outputs=[str(ADAPTER_DIR), str(LOG_DIR / 'train_metrics.json')], next_phase='8_merge_hf')
    record_disconnect_point('7_save_adapter', 'adapter_saved', adapter_path=str(ADAPTER_DIR))


In [ ]:
# Phase 8: merge LoRA adapter into HF/safetensors export
if phase_done('8_merge_hf') and has_nonzero_safetensors(MERGED_DIR):
    print('Merged export already exists:', MERGED_DIR)
elif not ADAPTER_DIR.exists():
    raise FileNotFoundError(f'Adapter missing: {ADAPTER_DIR}. Run Phase 6 first.')
else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    write_run_state('8_merge_hf', 'started', next_phase='9_generation_smoke')
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16 if torch.cuda.is_available() else torch.float32
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
    )
    merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size='2GB')
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR if (ADAPTER_DIR / 'tokenizer_config.json').exists() else BASE_MODEL)
    tokenizer.save_pretrained(MERGED_DIR)
    record_disconnect_point('8_merge_hf', 'merged_hf_export_saved', merged_hf_path=str(MERGED_DIR))
    mark_phase_done('8_merge_hf', outputs=[str(MERGED_DIR)], next_phase='9_generation_smoke')
    print('Merged HF export:', MERGED_DIR)


In [ ]:
# Phase 9: generation smoke on held-out rows
# This is not a replacement for Android LiteRT smoke; it catches obvious router collapse before conversion.
RUN_GENERATION_SMOKE = os.environ.get('RUN_GENERATION_SMOKE', '1') == '1'
SMOKE_ROWS_PER_TYPE = int(os.environ.get('SMOKE_ROWS_PER_TYPE', '4'))
SMOKE_MAX_NEW_TOKENS = int(os.environ.get('SMOKE_MAX_NEW_TOKENS', '220'))
SMOKE_REPORT = LOG_DIR / 'generation_smoke_v4_1.json'

def extract_json_object(text):
    start = text.find('{')
    end = text.rfind('}')
    if start < 0 or end < start:
        raise ValueError('no JSON object')
    return json.loads(text[start:end + 1])

if not RUN_GENERATION_SMOKE:
    print('RUN_GENERATION_SMOKE=False; skipping generation smoke')
elif not has_nonzero_safetensors(MERGED_DIR):
    raise FileNotFoundError(f'Merged export missing: {MERGED_DIR}')
else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        torch_dtype=dtype,
        device_map='auto' if torch.cuda.is_available() else None,
    )
    model.eval()

    payload = json.loads(DATASET.read_text(encoding='utf-8'))
    candidates = []
    for split_rows in payload['validation'].values():
        by_type = {}
        for row in split_rows:
            by_type.setdefault(row['row_type'], []).append(row)
        for rows_for_type in by_type.values():
            candidates.extend(rows_for_type[:SMOKE_ROWS_PER_TYPE])

    results = []
    for row in candidates:
        prompt = render_messages(row['messages'][:-1], include_assistant=True) if False else render_messages(row['messages'], include_assistant=False)
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=SMOKE_MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        expected = json.loads(row['messages'][-1]['content'])
        try:
            parsed = extract_json_object(generated)
            ok = parsed.get('function') == expected.get('function')
            err = ''
        except Exception as exc:
            parsed = {}
            ok = False
            err = str(exc)
        results.append({
            'row_type': row['row_type'],
            'expected_function': expected.get('function'),
            'predicted_function': parsed.get('function'),
            'ok': ok,
            'error': err,
            'raw': generated[:600],
        })
    summary = {
        'total': len(results),
        'exact_tool_accuracy': sum(1 for r in results if r['ok']) / len(results) if results else 0.0,
        'failures': [r for r in results if not r['ok']][:30],
    }
    atomic_write_json(SMOKE_REPORT, {'summary': summary, 'results': results})
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    record_event('9_generation_smoke', 'completed', **summary)
    record_disconnect_point('9_generation_smoke', 'generation_smoke_completed', report=str(SMOKE_REPORT), exact_tool_accuracy=summary['exact_tool_accuracy'])
    mark_phase_done('9_generation_smoke', outputs=[str(SMOKE_REPORT)], next_phase='10_done_marker')


In [ ]:
# Phase 10 / 11: done marker and local handoff copies
payload = json.loads(DATASET.read_text(encoding='utf-8')) if DATASET.exists() else {'train': [], 'validation': {}}
dataset_sha256 = file_sha256(DATASET) if DATASET.exists() else ''
status = 'complete' if ADAPTER_DIR.exists() and has_nonzero_safetensors(MERGED_DIR) else 'dataset_ready_or_training_incomplete'
smoke_summary = {}
if (LOG_DIR / 'generation_smoke_v4_1.json').exists():
    smoke_summary = json.loads((LOG_DIR / 'generation_smoke_v4_1.json').read_text(encoding='utf-8')).get('summary', {})

done = {
    'version': 'v4_1_senior_evidence_router',
    'run_name': RUN_NAME,
    'run_suffix': RUN_SUFFIX,
    'finished_at': utc_now(),
    'status': status,
    'last_completed_phase': last_completed_phase(),
    'dataset_path': str(DATASET),
    'dataset_sha256': dataset_sha256,
    'train_rows': len(payload.get('train', [])),
    'validation_rows': {k: len(v) for k, v in payload.get('validation', {}).items()},
    'model': BASE_MODEL,
    'checkpoint_dir': str(CKPT_DIR),
    'latest_checkpoint': latest_checkpoint(CKPT_DIR),
    'adapter_path': str(ADAPTER_DIR),
    'merged_hf_path': str(MERGED_DIR),
    'litertlm_path': str(TARGET_LITERTLM),
    'conversion_status': 'not_started',
    'tool_call_eval': 'finetune/metrics/tool_call_eval_v4_senior.json',
    'generation_smoke': str(LOG_DIR / 'generation_smoke_v4_1.json'),
    'generation_smoke_summary': smoke_summary,
    'resume_log': str(RUN_EVENTS),
    'disconnect_points': str(DISCONNECT_POINTS),
    'note': 'LiteRT conversion and Pixel smoke are separate acceptance gates.',
}
atomic_write_json(DONE_MARKER, done)
mark_phase_done('10_done_marker', outputs=[str(DONE_MARKER)], next_phase='litert_conversion_smoke')
record_disconnect_point('10_done_marker', 'done_marker_written', done_marker=str(DONE_MARKER), status=status)

# Repo-side copies are useful after downloading artifacts; ignore failures in Drive-only runs.
try:
    (REPO_DIR / 'finetune/metrics').mkdir(parents=True, exist_ok=True)
    atomic_write_json(REPO_DONE_COPY, done)
    if RUN_STATE.exists():
        shutil.copy2(RUN_STATE, REPO_RUN_STATE_COPY)
    if SMOKE_REPORT.exists():
        shutil.copy2(SMOKE_REPORT, REPO_DIR / 'finetune/metrics/generation_smoke_v4_1.json')
except Exception as exc:
    print('Repo metric copy skipped:', exc)

print(json.dumps(done, indent=2, ensure_ascii=False))
print('\nNext step: convert MERGED_DIR to LiteRT-LM in a separate conversion runtime, then Pixel smoke before enabling the backend.')
